# Notebook 2 — IC Toward a River / Lake Target (GEE/xee)
### GeomorphConn · Singh, Cavalli & Crema (2026)

This notebook computes **IC-target** relative to a river/lake target feature while keeping raster inputs cloud-native through GEE/xee.

### Inputs
- Required: target vector (shapefile/geojson/gpkg) to define target nodes
- Optional: explicit bounds tuple (otherwise target extent is used)
- Optional: local DEM override (if not provided, DEM is fetched from GEE)

### Workflow
1. Fetch DEM/NDVI/rainfall from GEE for the target region
2. Build Landlab grid and rasterize target feature
3. Run IC-outlet and IC-target across weight scenarios (DEM-only and combinations)
4. Compare and export results with matplotlib visualizations

```bash
pip install 'GeomorphConn[all]'
```

## 0 · Configuration

In [ ]:
from pathlib import Path

# ── Required target vector ───────────────────────────────────────────────────
TARGET_PATH = Path("data/river_or_lake.geojson")  # shapefile/geojson/gpkg/etc.
if not TARGET_PATH.exists():
    raise FileNotFoundError(f"Target vector not found: {TARGET_PATH}")

# ── Region bounds (optional) ─────────────────────────────────────────────────
# If None, fetch region is inferred from TARGET_PATH extent
BOUNDS = None  # e.g., (78.8, 29.6, 80.0, 30.5)

# ── Optional local DEM override (otherwise fetched from GEE) ────────────────
DEM_OVERRIDE_PATH = None  # e.g., Path("data/dem.tif")

# ── GEE configuration ────────────────────────────────────────────────────────
GEE_PROJECT = "drylands-aberuni"
DEM_SOURCE = "COPDEM30"
RF_SOURCE = "CHIRPS"
NDVI_SOURCE = "SENTINEL2"
START_DATE = "2020-05-01"
END_DATE = "2020-09-30"
SCALE = 30
CRS = "EPSG:4326"

# ── IC / weighting options ───────────────────────────────────────────────────
FLOW_DIRECTOR = "FlowDirectorDINF"
WEIGHT_SCENARIOS = [
    "roughness_only",
    "rainfall_only",
    "ndvi_only",
    "ndvi_rainfall",
    "ndvi_rainfall_roughness",
]
PLOT_SCENARIO = "ndvi_rainfall"

OUTPUT_DIR = Path("output_nb2")
OUTPUT_DIR.mkdir(exist_ok=True)

## 1 · Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from landlab import RasterModelGrid

from geomorphconn.components import ConnectivityIndex
from geomorphconn.utils import rasterize_targets
from geomorphconn.gee import GEEFetcher
from geomorphconn.weights import (
    WeightBuilder,
    NDVIWeight,
    RainfallWeight,
    preset_rainfall_ndvi,
    preset_rainfall_ndvi_roughness,
    preset_roughness_only,
)

print("All imports successful.")

## 2 · Fetch raster inputs from GEE (optional local DEM override)

In [ ]:
def _load_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float64)
        profile = src.profile.copy()
        if src.nodata is not None:
            arr[arr == src.nodata] = np.nan
    return arr, profile

bounds_input = BOUNDS if BOUNDS is not None else str(TARGET_PATH)

fetcher = GEEFetcher(
    bounds=bounds_input,
    dem_source=DEM_SOURCE,
    rainfall_source=RF_SOURCE,
    ndvi_source=NDVI_SOURCE,
    start_date=START_DATE,
    end_date=END_DATE,
    scale=SCALE,
    crs=CRS,
    gee_project=GEE_PROJECT,
)

dem, ndvi, rainfall, profile = fetcher.fetch()

if DEM_OVERRIDE_PATH is not None and Path(DEM_OVERRIDE_PATH).exists():
    dem, dem_profile = _load_raster(Path(DEM_OVERRIDE_PATH))
    profile.update({
        "transform": dem_profile["transform"],
        "width": dem_profile["width"],
        "height": dem_profile["height"],
        "crs": dem_profile["crs"],
    })
    print(f"Using DEM override: {DEM_OVERRIDE_PATH}")

nrows, ncols = dem.shape
dx = abs(profile["transform"].a)

import geopandas as gpd
target_gdf = gpd.read_file(TARGET_PATH)

print(f"Fetched rasters: {nrows}x{ncols}, dx={dx:.1f} m")

fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=100)
for ax, arr, title, cmap in zip(
    axes,
    [dem, ndvi, rainfall],
    [f"{DEM_SOURCE} DEM", f"NDVI ({NDVI_SOURCE})", f"Rainfall ({RF_SOURCE})"],
    ["terrain", "RdYlGn", "Blues"],
):
    im = ax.imshow(arr, cmap=cmap, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()

## 3 · Build Landlab grid

In [ ]:
def arr_to_node(arr):
    """Flip (GeoTIFF → Landlab) and ravel to 1D node array."""
    a = np.flipud(arr).copy()
    a[np.isnan(a)] = 0.0
    return a.ravel()

dem_ll = np.flipud(dem).copy()
dem_ll[np.isnan(dem_ll)] = np.nanmin(dem_ll) - 1.0

grid = RasterModelGrid((nrows, ncols), xy_spacing=dx)
grid.add_field("topographic__elevation", dem_ll.ravel(), at="node")

ndvi_ll     = arr_to_node(ndvi)
rainfall_ll = arr_to_node(rainfall)

print(f"Grid: {grid.number_of_nodes:,} nodes")

## 4 · Rasterize target features

`rasterize_targets` converts the vector river/lake geometry to a list of Landlab node IDs, replicating ArcGIS *PolygonToRaster → SetNull → impose-on-FDR* steps in one call.

In [ ]:
target_nodes = rasterize_targets(
    source        = target_gdf,
    grid          = grid,
    dem_transform = profile["transform"],
    dem_crs       = profile["crs"],
    buffer_m      = dx,    # 1-cell buffer to ensure narrow lines are captured
)

print(f"Target nodes: {len(target_nodes):,}  "
      f"({100*len(target_nodes)/grid.number_of_nodes:.1f}% of grid)")

# Visualise target overlay on DEM
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
ax.imshow(dem, cmap="terrain", aspect="equal", interpolation="nearest")
# Mark target cells
target_mask_2d = np.zeros(grid.number_of_nodes, dtype=bool)
target_mask_2d[target_nodes] = True
ax.imshow(np.flipud(target_mask_2d.reshape(nrows, ncols)),
          cmap="cool", alpha=0.6, aspect="equal", interpolation="nearest")
ax.set_title("DEM with target nodes highlighted"); ax.axis("off")
plt.tight_layout(); plt.show()

## 5 · Run IC-outlet and IC-target across weight scenarios

In [ ]:
import time

def make_weight_builder(name, rainfall_nodes, ndvi_nodes, grid_obj):
    if name == "roughness_only":
        return preset_roughness_only(grid_obj)
    if name == "rainfall_only":
        return WeightBuilder().add(RainfallWeight(rainfall_nodes))
    if name == "ndvi_only":
        return WeightBuilder().add(NDVIWeight(ndvi_nodes))
    if name == "ndvi_rainfall":
        return preset_rainfall_ndvi(rainfall_nodes, ndvi_nodes)
    if name == "ndvi_rainfall_roughness":
        return preset_rainfall_ndvi_roughness(rainfall_nodes, ndvi_nodes, grid_obj)
    raise ValueError(f"Unknown scenario: {name}")

ic_outlet_results = {}
ic_target_results = {}

for scenario in WEIGHT_SCENARIOS:
    wb = make_weight_builder(scenario, rainfall_ll, ndvi_ll, grid)

    # outlet
    grid.at_node["topographic__elevation"][:] = dem_ll.ravel()
    ic_out = ConnectivityIndex(grid, flow_director=FLOW_DIRECTOR, weight=wb)
    t0 = time.time()
    ic_out.run_one_step()
    t_out = time.time() - t0
    ic_outlet_results[scenario] = {
        "IC": ic_out.as_2d(),
        "W": ic_out.as_2d("connectivity_index__W"),
        "Dup": ic_out.as_2d("connectivity_index__Dup"),
        "Ddn": ic_out.as_2d("connectivity_index__Ddn"),
        "runtime_s": t_out,
    }

    # target
    grid.at_node["topographic__elevation"][:] = dem_ll.ravel()
    ic_tgt = ConnectivityIndex(
        grid,
        flow_director=FLOW_DIRECTOR,
        weight=wb,
        target_nodes=target_nodes,
    )
    t0 = time.time()
    ic_tgt.run_one_step()
    t_tgt = time.time() - t0
    ic_target_results[scenario] = {
        "IC": ic_tgt.as_2d(),
        "W": ic_tgt.as_2d("connectivity_index__W"),
        "Dup": ic_tgt.as_2d("connectivity_index__Dup"),
        "Ddn": ic_tgt.as_2d("connectivity_index__Ddn"),
        "runtime_s": t_tgt,
    }

summary_rows = []
for scenario in WEIGHT_SCENARIOS:
    v_out = ic_outlet_results[scenario]["IC"]
    v_tgt = ic_target_results[scenario]["IC"]
    v_out = v_out[np.isfinite(v_out)]
    v_tgt = v_tgt[np.isfinite(v_tgt)]
    summary_rows.append({
        "scenario": scenario,
        "outlet_median": float(np.median(v_out)),
        "target_median": float(np.median(v_tgt)),
        "delta_median": float(np.median(v_tgt) - np.median(v_out)),
        "runtime_outlet_s": float(ic_outlet_results[scenario]["runtime_s"]),
        "runtime_target_s": float(ic_target_results[scenario]["runtime_s"]),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("scenario")
display(summary_df.round(4))

## 6 · Compare IC-outlet vs IC-target

In [ ]:
IC_outlet = ic_outlet_results[PLOT_SCENARIO]["IC"]
IC_target = ic_target_results[PLOT_SCENARIO]["IC"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=120)
fig.suptitle(
    f"IC-outlet vs IC-target (scenario: {PLOT_SCENARIO})",
    fontweight="bold"
 )

all_vals = np.concatenate([IC_outlet[np.isfinite(IC_outlet)], IC_target[np.isfinite(IC_target)]])
vmin, vmax = np.nanpercentile(all_vals, 2), np.nanpercentile(all_vals, 98)

for ax, arr, title in zip(
    axes,
    [IC_outlet, IC_target, IC_target - IC_outlet],
    ["IC - outlet", "IC - target", "Delta IC (target - outlet)"],
):
    kw = dict(interpolation="nearest", aspect="equal")
    if "Delta" in title:
        lim = max(abs(np.nanpercentile(arr, 2)), abs(np.nanpercentile(arr, 98)))
        im = ax.imshow(arr, cmap="RdBu_r", vmin=-lim, vmax=lim, **kw)
    else:
        im = ax.imshow(arr, cmap="RdYlGn", vmin=vmin, vmax=vmax, **kw)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "IC_outlet_vs_target.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Scenario-level summary plot (matplotlib)
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=120)

x = np.arange(len(summary_df))
width = 0.36
ax.bar(x - width/2, summary_df["outlet_median"], width=width, label="Outlet median", color="#3f7f93")
ax.bar(x + width/2, summary_df["target_median"], width=width, label="Target median", color="#cc7a00")

ax.set_xticks(x)
ax.set_xticklabels(summary_df["scenario"], rotation=25, ha="right")
ax.set_ylabel("Median IC")
ax.set_title("Median IC by weight scenario")
ax.legend()
plt.tight_layout()
plt.show()

display(summary_df.round(4))
summary_df.to_csv(OUTPUT_DIR / "weight_scenarios_summary.csv", index=False)

## 7 · Export GeoTIFFs

In [ ]:
def save_tiff(arr, path, profile):
    out = arr.astype(np.float32)
    out[~np.isfinite(out)] = -9999.0
    p   = profile.copy()
    p.update(dtype="float32", count=1, nodata=-9999.0, compress="lzw")
    with rasterio.open(path, "w", **p) as dst:
        dst.write(out, 1)
    print(f"  ✓ {path}")

save_tiff(IC_outlet,                          OUTPUT_DIR / f"IC_outlet_{PLOT_SCENARIO}.tif", profile)
save_tiff(IC_target,                          OUTPUT_DIR / f"IC_target_{PLOT_SCENARIO}.tif", profile)
save_tiff(IC_target - IC_outlet,              OUTPUT_DIR / f"IC_delta_{PLOT_SCENARIO}.tif",  profile)
save_tiff(ic_target_results[PLOT_SCENARIO]["W"],   OUTPUT_DIR / f"W_{PLOT_SCENARIO}.tif",   profile)
save_tiff(ic_target_results[PLOT_SCENARIO]["Dup"], OUTPUT_DIR / f"Dup_{PLOT_SCENARIO}.tif", profile)
save_tiff(ic_target_results[PLOT_SCENARIO]["Ddn"], OUTPUT_DIR / f"Ddn_{PLOT_SCENARIO}.tif", profile)

summary_df.to_csv(OUTPUT_DIR / "weight_scenarios_summary.csv", index=False)
print("\nAll outputs saved to:", OUTPUT_DIR.resolve())

## 8 · Notes on IC-target interpretation

**IC-outlet** quantifies how connected a cell is to the basin's natural outlet.  
**IC-target** quantifies how connected a cell is to the *nearest point on the river network* (or any other target feature).

For hillslope–channel coupling studies, IC-target is more physically meaningful because:
- Sediment eroded from a hillslope typically reaches a channel long before it reaches the basin outlet.
- The difference map (Δ IC = IC-target − IC-outlet) highlights cells where the presence of the river network *significantly alters* apparent connectivity.

In ArcGIS Pro, this workflow is available in `ConnectivityTools.atbx` using `ICtargetwithNDVIRFweightCalc` (`arcgis_tools/`).  
Both tools are described in Singh, Cavalli & Crema (2026), *JOSS*.